# 3.6 章节练习参考答案

[客观题参考答案与解析](./03.06_answer.txt)

## 实验题参考代码

### 实验代码示例

下面的代码分别运行 `clone_output=False` 和 `clone_output=True`。请重点观察第一次输出在第二次 Replay 后是否发生变化，并记录稳定 Replay 耗时与显存增量。

In [ ]:
import time
import torch
import torch_npu

assert torch.npu.is_available(), "请在已安装 CANN 和 torch_npu 的昇腾环境中运行"
torch.manual_seed(0)


class OutputReuseModel(torch.nn.Module):
    def forward(self, x):
        return torch.relu(x * 2 + 1)


# 先预热以排除首次编译与 Capture，再统计稳定 Replay 的平均耗时。
def benchmark(fn, x, warmup=3, iters=20):
    for _ in range(warmup):
        fn(x)
    torch.npu.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        fn(x)
    torch.npu.synchronize()
    return (time.perf_counter() - start) * 1000 / iters


# 每种 clone_output 配置独立编译，避免两组实验相互影响。
def run_case(clone_output):
    torch.npu.empty_cache()
    model = OutputReuseModel().npu().eval()
    compiled = torch.compile(
        model,
        backend="npugraph_ex",
        options={"clone_output": clone_output},
        fullgraph=True,
        dynamic=False,
    )

    x1 = torch.randn(1024, 1024, dtype=torch.float16).npu()
    x2 = torch.randn(1024, 1024, dtype=torch.float16).npu()
    memory_before = torch.npu.memory_allocated()

    with torch.no_grad():
        # 保存第一次输出快照，再用不同输入触发第二次 Replay。
        first_output = compiled(x1)
        first_snapshot = first_output.clone()
        compiled(x2)  # 触发第二次 Replay
        torch.npu.synchronize()
        output_changed = not torch.equal(first_output, first_snapshot)
        # 使用 Eager 结果作为精度基线，确认快照数值正确。
        eager_output = model(x1)
        torch.testing.assert_close(first_snapshot, eager_output)
        replay_ms = benchmark(compiled, x1)

    memory_delta = torch.npu.memory_allocated() - memory_before
    return output_changed, memory_delta / 1024**2, replay_ms


for enabled in (False, True):
    changed, memory_mb, replay_ms = run_case(enabled)
    print(
        f"clone_output={enabled:<5} | 第一次输出是否变化: {changed} "
        f"| 显存增量: {memory_mb:.2f} MiB | Replay: {replay_ms:.3f} ms"
    )
